<a href="https://colab.research.google.com/github/jagan631/GenAI/blob/main/GenAI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.8 MB/s eta 0

In [3]:
import os
import textwrap

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

from pypdf import PdfReader
from google.colab import files


In [5]:
# Create a folder for docs
os.makedirs("docs", exist_ok=True)

print("Please select your PDF files...")
uploaded = files.upload()

for filename, filedata in uploaded.items():
    path = os.path.join("docs", filename)
    with open(path, "wb") as f:
        f.write(filedata)
    print("Saved:", path)


Please select your PDF files...


Saving womens writing.pdf to womens writing.pdf
Saved: docs/womens writing.pdf


In [6]:
def extract_text_from_pdfs(folder="docs"):
    all_text = ""
    for fname in os.listdir(folder):
        if fname.lower().endswith(".pdf"):
            path = os.path.join(folder, fname)
            print("Reading:", path)
            reader = PdfReader(path)
            for page in reader.pages:
                text = page.extract_text() or ""
                all_text += text + "\n"
    return all_text

raw_text = extract_text_from_pdfs("docs")
print("Total characters:", len(raw_text))
print(raw_text[:1000])  # preview


Reading: docs/womens writing.pdf
Total characters: 16627
1. Eve to Her Daughters by Judith Wright 
Introduction 
Judith Wright, one of Australia’s most celebrated poets, is known for blending themes of 
history, feminism, ecology, and human identity. In her poem “Eve to Her Daughters”, Wright 
retells the Biblical myth of Adam and Eve, but through Eve’s own voice. By doing so, she 
shifts the focus from the patriarchal perspective of Genesis to a female-centered 
interpretation. The poem becomes not just a story of humanity’s beginning, but also a 
commentary on modern science, technology, and the survival of mankind. 
Theme of Rationality vs Imagination 
The poem sets up a sharp contrast between Adam and Eve. Adam is depicted as restless, 
rational, and scientific. He represents the modern human mind obsessed with invention, 
exploration, and control. His search for knowledge eventually leads him away from nature 
and into the realm of technology. 
Eve, on the other hand, symbolizes i

In [7]:
def create_chunks(text, max_chars=800, overlap=100):
    """
    Split text into overlapping chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    # Remove very tiny chunks
    chunks = [c for c in chunks if len(c) > 50]
    return chunks

chunks = create_chunks(raw_text)
print("Number of chunks:", len(chunks))
print("Sample chunk:\n", chunks[0][:500])


Number of chunks: 24
Sample chunk:
 1. Eve to Her Daughters by Judith Wright 
Introduction 
Judith Wright, one of Australia’s most celebrated poets, is known for blending themes of 
history, feminism, ecology, and human identity. In her poem “Eve to Her Daughters”, Wright 
retells the Biblical myth of Adam and Eve, but through Eve’s own voice. By doing so, she 
shifts the focus from the patriarchal perspective of Genesis to a female-centered 
interpretation. The poem becomes not just a story of humanity’s beginning, but also a 
co


In [8]:
# Load sentence embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Compute embeddings
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

# Build FAISS index
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(chunk_embeddings)

print("FAISS index built with", index.ntotal, "vectors.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built with 24 vectors.


In [9]:
model_name = "google/flan-t5-base"

hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text2text = pipeline(
    "text2text-generation",
    model=hf_model,
    tokenizer=hf_tokenizer
)

print("Model loaded:", model_name)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


Model loaded: google/flan-t5-base


In [10]:
def retrieve_context(query, top_k=3):
    # Embed the question
    q_emb = embed_model.encode([query]).astype("float32")
    # Search in FAISS index
    distances, indices = index.search(q_emb, top_k)

    retrieved_chunks = []
    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])
    return retrieved_chunks


def build_prompt(query, context_chunks):
    context_text = "\n\n".join(context_chunks)
    prompt = f"""
You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context_text}

Question: {query}
Answer:
"""
    return prompt.strip()


def answer_question(query, top_k=3, max_new_tokens=256):
    context_chunks = retrieve_context(query, top_k=top_k)

    print("🔍 Retrieved context snippets:\n")
    for i, c in enumerate(context_chunks, 1):
        print(f"--- Chunk {i} ---")
        print(textwrap.shorten(c.replace("\n", " "), width=300, placeholder="..."))
        print()

    prompt = build_prompt(query, context_chunks)
    result = text2text(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"]
    return result


In [12]:
question = "What does the deer symbolize in the story In a Forest, A Deer?"
answer = answer_question(question)
print("\n💬 Answer:\n", answer)


🔍 Retrieved context snippets:

--- Chunk 1 ---
strong feminist writer, and in this story, she highlights the conflict between women’s inner desires and societal expectations. The female voice in the story reveals the silent suffering of countless women whose individuality is ignored. Through her portrayal, Ambai questions the patriarchal...

--- Chunk 2 ---
ions yet yearn for freedom. The deer in the forest becomes a powerful metaphor for women’s vulnerability, innocence, and desire for liberation. Theme of Women’s Struggles The central theme of the story is the oppression of women in a male-dominated world. The protagonist, like many women, faces...

--- Chunk 3 ---
e and questions the patriarchal blame placed on women. The poem also warns against blind faith in science and technology, urging balance, harmony with nature, and survival through acceptance and wisdom. Thus, the poem becomes both feminist and ecological in message, reminding us that true...


💬 Answer:
 innocence, beauty,